# Road Following Live (Autonomous Simulator Mode)

This notebook runs the trained ResNet-18 Road Following model autonomously inside the **DonkeyCar Simulator (`CarSimulator`)**.

### 1. Setup Environment & Load Trained Model

In [1]:
import os
import sys
from pathlib import Path
import torch
import torchvision

# Add simulation root to sys.path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from utils import preprocess
from xy_dataset import XYDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load model architecture (ResNet-18)
model = torchvision.models.resnet18(pretrained=False)
model.fc = torch.nn.Linear(512, 2)  # x, y coordinates

model_path = 'road_following_model.pth'
if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"Successfully loaded model weights from '{model_path}'")
else:
    print(f"Warning: '{model_path}' not found! Please train and save model first in interactive_regression.ipynb")

model = model.to(device).eval()


d:\Users\nhatt\AppData\Local\Programs\Python\Python314\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\Users\nhatt\AppData\Local\Programs\Python\Python314\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Successfully loaded model weights from 'road_following_model.pth'


### 2. Initialize Simulator (`CarSimulator`)

In [2]:
from simulation import CarSimulator, bgr8_to_jpeg

# Reuse or close previous CarSimulator instance
if 'Car' in globals() and Car is not None:
    try:
        Car.close()
    except Exception:
        pass

Car = CarSimulator()
print("Simulator initialized successfully!")


INFO:gym_donkeycar.core.client:connecting to localhost:9091 
d:\Users\nhatt\AppData\Local\Programs\Python\Python314\Lib\site-packages\gymnasium\spaces\box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
d:\Users\nhatt\AppData\Local\Programs\Python\Python314\Lib\site-packages\gymnasium\spaces\box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
INFO:gym_donkeycar.envs.donkey_sim:on need car config
INFO:gym_donkeycar.envs.donkey_sim:sending car config.
INFO:gym_donkeycar.envs.donkey_sim:sim started!


starting DonkeyGym env
Setting default: start_delay 5.0
Setting default: max_cte 8.0
Setting default: frame_skip 1
Setting default: cam_resolution (120, 160, 3)
Setting default: log_level 20
Setting default: host localhost
Setting default: port 9091
Setting default: steer_limit 1.0
Setting default: throttle_min 0.0
Setting default: throttle_max 1.0
Simulator initialized successfully!


### 3. Basic Controller (P-Gain + Bias)

In [3]:
import cv2
import ipywidgets
import traitlets
import threading
import time
from IPython.display import display
from ipywidgets import Layout

slider_style = {'description_width': '140px'}

# Control Sliders with numeric readout display enabled
network_output_slider = ipywidgets.FloatSlider(description='Network Output', min=-1.0, max=1.0, value=0.0, step=0.01, readout=True, readout_format='.2f', disabled=True, layout=Layout(width='400px'), style=slider_style)
steering_gain_slider  = ipywidgets.FloatSlider(description='Steering Gain', min=-2.0, max=2.0, value=1.0, step=0.05, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=slider_style)
steering_bias_slider  = ipywidgets.FloatSlider(description='Steering Bias', min=-0.5, max=0.5, value=0.0, step=0.01, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=slider_style)
steering_value_slider = ipywidgets.FloatSlider(description='Final Steering', min=-1.0, max=1.0, value=0.0, step=0.01, readout=True, readout_format='.2f', disabled=True, layout=Layout(width='400px'), style=slider_style)
throttle_slider       = ipywidgets.FloatSlider(description='Throttle', min=-1.0, max=1.0, value=0.15, step=0.01, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=slider_style)

# Live Stream & Prediction Preview Widget
state_widget = ipywidgets.ToggleButtons(options=['Off', 'Autonomous Live'], description='Mode', value='Off')
prediction_widget = ipywidgets.Image(value=Car.value, format='jpeg', width=Car.obs.shape[1], height=Car.obs.shape[0])
reset_button = ipywidgets.Button(description='Reset Track', button_style='warning', icon='refresh')

live_active = False

def live_drive_loop():
    global live_active
    while live_active:
        try:
            image = Car.obs
            preprocessed = preprocess(image)
            
            with torch.no_grad():
                output = model(preprocessed).detach().cpu().numpy().flatten()
            
            x = float(output[0])
            y = float(output[1]) if len(output) > 1 else 0.0
            
            network_output_slider.value = x
            
            # Calculate final steering value with Gain & Bias
            steering = x * steering_gain_slider.value + steering_bias_slider.value
            steering = max(-1.0, min(1.0, steering))
            steering_value_slider.value = steering
            
            # Step the car in DonkeyCar simulator
            Car.run(steering, throttle_slider.value)
            
            # Draw predicted target dot (blue circle) on video preview
            px = int(Car.obs.shape[1] * (x / 2.0 + 0.5))
            py = int(Car.obs.shape[0] * (y / 2.0 + 0.5))
            
            prediction = Car.obs.copy()
            prediction = cv2.circle(prediction, (px, py), 8, (255, 0, 0), 3)
            prediction_widget.value = bgr8_to_jpeg(prediction)
            
        except Exception as e:
            print(f"Error in live drive loop: {e}")
            break
            
        time.sleep(0.05)  # 20 FPS loop

def on_state_change(change):
    global live_active
    if change['new'] == 'Autonomous Live':
        if not live_active:
            live_active = True
            t = threading.Thread(target=live_drive_loop, daemon=True)
            t.start()
    else:
        live_active = False

state_widget.observe(on_state_change, names='value')

def on_reset_clicked(b):
    global live_active
    state_widget.value = 'Off'
    live_active = False
    time.sleep(0.1)
    Car.reset()
    prediction_widget.value = Car.value

reset_button.on_click(on_reset_clicked)

# Clean Non-Overlapping Layout
center_box = ipywidgets.VBox([
    prediction_widget,
    ipywidgets.HBox([state_widget, reset_button])
], layout=Layout(align_items='center', margin='0px 0px 15px 0px'))

sliders_box = ipywidgets.VBox([
    network_output_slider,
    steering_gain_slider,
    steering_bias_slider,
    steering_value_slider,
    throttle_slider
], layout=Layout(align_items='center'))

display(ipywidgets.VBox([center_box, sliders_box]))


### 4. Advanced Steering Controllers (PID & Stanley Controller with 1D Kalman Filtering)

This section provides two advanced autonomous steering controllers:

1. **Stanley Controller (Recommended - Step 1 Upgrade):**
   * Uses **Heading Error ($\psi$)** estimated from Kalman lateral velocity + **Cross-Track Error ($	ext{CTE} = x$)**.
   * Steering formula: $\delta = \psi + \arctan\left(\frac{k \cdot \text{CTE}}{v + \epsilon}\right) + \text{bias}$.
   * Corrects heading and lateral offset simultaneously without needing data retraining!

2. **PID Controller:**
   * Traditional Proportional, Integral ($K_i$), and Derivative ($K_d$) controller with anti-windup clamping.

3. **1D Kalman Filtering:**
   * Smooths raw neural network predictions and estimates true lateral velocity $v_x = \frac{dx}{dt}$ without noise spikes.

In [ ]:
import cv2
import ipywidgets
import traitlets
import threading
import time
import json
from IPython.display import display
from ipywidgets import Layout
import Controller

# Initialize Controllers from Controller.py
pid = Controller.PIDController()
stanley = Controller.StanleyController()

pid_style = {'description_width': '140px'}

# Controller Sliders
k_stanley_slider = ipywidgets.FloatSlider(description='Stanley Gain (k)', min=0.1, max=3.0, value=1.2, step=0.05, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=pid_style)
kp_slider        = ipywidgets.FloatSlider(description='Kp (PID Gain)', min=0.0, max=3.0, value=1.0, step=0.05, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=pid_style)
ki_slider        = ipywidgets.FloatSlider(description='Ki (Integral)', min=0.0, max=0.5, value=0.0, step=0.01, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=pid_style)
kd_slider        = ipywidgets.FloatSlider(description='Kd (Damping)', min=0.0, max=1.0, value=0.15, step=0.02, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=pid_style)
alpha_slider     = ipywidgets.FloatSlider(description='Alpha (Filter)', min=0.1, max=1.0, value=0.7, step=0.05, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=pid_style)
pid_bias_slider  = ipywidgets.FloatSlider(description='Steering Bias', min=-0.5, max=0.5, value=0.0, step=0.01, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=pid_style)

base_throttle_slider = ipywidgets.FloatSlider(description='Base Throttle', min=0.05, max=0.5, value=0.20, step=0.01, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=pid_style)
brake_gain_slider    = ipywidgets.FloatSlider(description='Brake Gain', min=0.0, max=0.4, value=0.10, step=0.01, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=pid_style)

pid_steering_disp = ipywidgets.FloatSlider(description='Live Steering', min=-1.0, max=1.0, value=0.0, step=0.01, readout=True, readout_format='.2f', disabled=True, layout=Layout(width='400px'), style=pid_style)
pid_throttle_disp = ipywidgets.FloatSlider(description='Live Throttle', min=0.0, max=0.5, value=0.15, step=0.01, readout=True, readout_format='.2f', disabled=True, layout=Layout(width='400px'), style=pid_style)

pid_state_widget       = ipywidgets.ToggleButtons(options=['Off', 'Stanley Live Drive', 'PID Live Drive'], description='Control Mode', value='Off')
pid_prediction_widget = ipywidgets.Image(value=Car.value, format='jpeg', width=Car.obs.shape[1], height=Car.obs.shape[0])
pid_reset_button       = ipywidgets.Button(description='Reset Track & Controller', button_style='warning', icon='refresh')
load_best_btn          = ipywidgets.Button(description='Load Best Config', button_style='info', icon='download')

pid_live_active = False

def load_best_config(b=None):
    cfg_path = 'best_pid_config.json'
    if os.path.exists(cfg_path):
        with open(cfg_path, 'r', encoding='utf-8') as f:
            cfg = json.load(f)
        kp_slider.value            = cfg.get('kp', kp_slider.value)
        ki_slider.value            = cfg.get('ki', ki_slider.value)
        kd_slider.value            = cfg.get('kd', kd_slider.value)
        alpha_slider.value         = cfg.get('alpha', alpha_slider.value)
        pid_bias_slider.value      = cfg.get('bias', pid_bias_slider.value)
        base_throttle_slider.value = cfg.get('base_throttle', base_throttle_slider.value)
        brake_gain_slider.value    = cfg.get('brake_gain', brake_gain_slider.value)
        print(f"Loaded best config from '{cfg_path}'!")
    else:
        print(f"Config file '{cfg_path}' not found!")

load_best_btn.on_click(load_best_config)

def pid_drive_loop():
    global pid_live_active
    pid.reset()
    stanley.reset()
    while pid_live_active:
        try:
            image = Car.obs
            preprocessed = preprocess(image)
            
            with torch.no_grad():
                output = model(preprocessed).detach().cpu().numpy().flatten()
            
            raw_x = float(output[0])
            raw_y = float(output[1]) if len(output) > 1 else 0.0
            
            mode = pid_state_widget.value
            if mode == 'Stanley Live Drive':
                steering, dyn_throttle = stanley.update(
                    raw_x=raw_x,
                    k=k_stanley_slider.value,
                    base_throttle=base_throttle_slider.value,
                    brake_gain=brake_gain_slider.value,
                    bias=pid_bias_slider.value,
                    alpha=alpha_slider.value
                )
                smoothed_x = stanley.smoothed_x
            else:
                steering = pid.update(
                    raw_x=raw_x,
                    kp=kp_slider.value,
                    ki=ki_slider.value,
                    kd=kd_slider.value,
                    alpha=alpha_slider.value,
                    bias=pid_bias_slider.value
                )
                dyn_throttle = base_throttle_slider.value - brake_gain_slider.value * abs(steering)
                dyn_throttle = max(0.05, min(0.5, dyn_throttle))
                smoothed_x = pid.smoothed_x
            
            pid_steering_disp.value = steering
            pid_throttle_disp.value = dyn_throttle
            
            # Step DonkeyCar simulator
            Car.run(steering, dyn_throttle)
            
            # Draw prediction dot (green circle for target)
            px = int(Car.obs.shape[1] * (smoothed_x / 2.0 + 0.5))
            py = int(Car.obs.shape[0] * (raw_y / 2.0 + 0.5))
            
            prediction = Car.obs.copy()
            prediction = cv2.circle(prediction, (px, py), 8, (0, 255, 0), 3)
            pid_prediction_widget.value = bgr8_to_jpeg(prediction)
            
        except Exception as e:
            print(f"Error in drive loop: {e}")
            break
            
        time.sleep(0.05)  # 20 FPS loop

def on_pid_state_change(change):
    global pid_live_active
    if change['new'] in ['PID Live Drive', 'Stanley Live Drive']:
        if not pid_live_active:
            pid_live_active = True
            t = threading.Thread(target=pid_drive_loop, daemon=True)
            t.start()
    else:
        pid_live_active = False

pid_state_widget.observe(on_pid_state_change, names='value')

def on_pid_reset_clicked(b):
    global pid_live_active
    pid_state_widget.value = 'Off'
    pid_live_active = False
    time.sleep(0.1)
    pid.reset()
    stanley.reset()
    Car.reset()
    pid_prediction_widget.value = Car.value

pid_reset_button.on_click(on_pid_reset_clicked)

# Clean Non-Overlapping Layout
pid_center_box = ipywidgets.VBox([
    pid_prediction_widget,
    ipywidgets.HBox([pid_state_widget, pid_reset_button, load_best_btn])
], layout=Layout(align_items='center', margin='0px 0px 15px 0px'))

pid_tuning_box = ipywidgets.VBox([
    ipywidgets.HTML(value="<h4>Steering Tuning (Stanley & PID)</h4>"),
    k_stanley_slider,
    kp_slider,
    ki_slider,
    kd_slider,
    alpha_slider,
    pid_bias_slider
], layout=Layout(margin='0px 20px 0px 0px'))

throttle_tuning_box = ipywidgets.VBox([
    ipywidgets.HTML(value="<h4>Dynamic Throttle & Outputs</h4>"),
    base_throttle_slider,
    brake_gain_slider,
    ipywidgets.HTML(value="<b>Live Outputs:</b>"),
    pid_steering_disp,
    pid_throttle_disp
])

controls_grid = ipywidgets.HBox([pid_tuning_box, throttle_tuning_box], layout=Layout(justify_content='space-around'))

display(ipywidgets.VBox([pid_center_box, controls_grid]))


### 5. Bayesian Hyperparameter Optimizer (Optuna TPE)

Optimizes PID and Kalman Filter parameters ($K_p, K_i, K_d, R_{\text{measure}}, K_{\text{brake}}$) using

In [5]:
import json
import time
import os
import torch
import optuna
import numpy as np
import ipywidgets
from IPython.display import display, clear_output

# Tuning settings
N_TRIALS          = 60    # Total Optuna trials to run
EPISODES_PER_TRIAL = 3   # Runs per config (averaged to reduce noise)
MAX_STEPS         = 200  # Max simulator steps per episode
BASE_THROTTLE     = 0.20
CONFIG_SAVE_PATH  = 'best_pid_config.json'

optuna.logging.set_verbosity(optuna.logging.WARNING)  # suppress verbose logs

# Progress widgets
trial_progress = ipywidgets.IntProgress(
    min=0, max=N_TRIALS, description='Trials:',
    bar_style='info', layout=ipywidgets.Layout(width='450px'),
    style={'description_width': '60px'})

best_score_label = ipywidgets.Label(value='Best score: --')
best_params_out  = ipywidgets.Output()
trial_log_out    = ipywidgets.Output(layout=ipywidgets.Layout(height='200px', overflow_y='auto', border='1px solid #ddd'))

display(ipywidgets.VBox([
    ipywidgets.HBox([trial_progress, best_score_label]),
    ipywidgets.HTML('<b>Best config so far:</b>'),
    best_params_out,
    ipywidgets.HTML('<b>Trial log:</b>'),
    trial_log_out
]))


# Evaluation function 
def evaluate_pid(kp, ki, kd, alpha, bias, brake_gain, trial=None):
    """
    Runs EPISODES_PER_TRIAL independent episodes and returns the mean score.
    Score = -mean_abs_x_deviation * 10 + survival_bonus
    (Lower deviation = better; longer survival = bonus)
    """
    episode_scores = []

    for ep in range(EPISODES_PER_TRIAL):
        Car.reset()
        pid.reset()

        x_deviations = []
        steps = 0

        for step in range(MAX_STEPS):
            image = Car.obs
            preprocessed = preprocess(image)

            with torch.no_grad():
                output = model(preprocessed).detach().cpu().numpy().flatten()

            raw_x = float(output[0])
            x_deviations.append(abs(raw_x))

            steering   = pid.update(raw_x=raw_x, kp=kp, ki=ki, kd=kd, alpha=alpha, bias=bias)
            dyn_throttle = max(0.05, min(0.5, BASE_THROTTLE - brake_gain * abs(steering)))

            result     = Car.run(steering, dyn_throttle)
            terminated = result.get('terminated', False)
            truncated  = result.get('truncated', False)
            steps += 1

            # Optuna intermediate pruning: report after 1st episode mid-point
            if trial is not None and ep == 0 and step == MAX_STEPS // 2:
                intermediate = -float(np.mean(x_deviations)) * 10 + steps * 0.3
                trial.report(intermediate, step)
                if trial.should_prune():
                    raise optuna.exceptions.TrialPruned()

            if terminated or truncated:
                break

        mean_dev = float(np.mean(x_deviations)) if x_deviations else 1.0
        # Score: penalize deviation, reward long survival
        # Negate because Optuna minimizes by default (we maximize score)
        ep_score = -mean_dev * 10 + steps * 0.3
        episode_scores.append(ep_score)

    return float(np.mean(episode_scores))


# Optuna objective
best_so_far = {'score': -float('inf'), 'config': None}

def objective(trial):
    kp         = trial.suggest_float('kp',         0.3,  2.5)
    ki         = trial.suggest_float('ki',         0.0,  0.3)
    kd         = trial.suggest_float('kd',         0.0,  0.5)
    alpha      = trial.suggest_float('alpha',      0.3,  1.0)
    bias       = trial.suggest_float('bias',      -0.2,  0.2)
    brake_gain = trial.suggest_float('brake_gain', 0.0,  0.35)

    score = evaluate_pid(kp, ki, kd, alpha, bias, brake_gain, trial=trial)

    trial_progress.value = trial.number + 1

    with trial_log_out:
        print(f"Trial {trial.number+1:03d}/{N_TRIALS} | "
              f"Kp={kp:.2f} Ki={ki:.2f} Kd={kd:.2f} "
              f"α={alpha:.2f} bias={bias:.2f} brake={brake_gain:.2f} "
              f"→ score={score:.3f}")

    # Track best
    if score > best_so_far['score']:
        best_so_far['score'] = score
        best_so_far['config'] = dict(
            kp=kp, ki=ki, kd=kd, alpha=alpha,
            bias=bias, base_throttle=BASE_THROTTLE,
            brake_gain=brake_gain, score=score
        )
        best_score_label.value = f'Best score: {score:.3f}'
        with best_params_out:
            clear_output(wait=True)
            print(json.dumps(best_so_far['config'], indent=2))

        # Save immediately whenever we find a better config
        with open(CONFIG_SAVE_PATH, 'w', encoding='utf-8') as f:
            json.dump(best_so_far['config'], f, indent=2)

    # Optuna minimizes → negate
    return -score


# Run study
study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=50)
)

print(f"Starting Optuna Bayesian Optimization: {N_TRIALS} trials × {EPISODES_PER_TRIAL} episodes each")
print(f"Search space: Kp∈[0.3,2.5], Ki∈[0,0.3], Kd∈[0,0.5], α∈[0.3,1.0], bias∈[-0.2,0.2], brake∈[0,0.35]\n")

study.optimize(objective, n_trials=N_TRIALS, catch=(Exception,))

# Print final summary
print("\n" + "="*60)
print(f" Optimization complete! Best trial: #{study.best_trial.number + 1}")
print(f"   Score (negated Optuna value): {-study.best_value:.4f}")
print(json.dumps(best_so_far['config'], indent=2))
print(f"\nBest config saved to '{CONFIG_SAVE_PATH}'")
print("Click 'Load Best Config' in Section 4 to apply it to the PID sliders.")


INFO:simulation:steering=-0.82 | throttle=0.05 | reward=0.817
INFO:simulation:steering=-1.00 | throttle=0.05 | reward=0.759
INFO:simulation:steering=-1.00 | throttle=0.05 | reward=0.706
INFO:simulation:steering=-0.93 | throttle=0.05 | reward=0.657
INFO:simulation:steering=-0.80 | throttle=0.06 | reward=0.606
INFO:simulation:steering=-0.71 | throttle=0.07 | reward=0.582
INFO:simulation:steering=-0.66 | throttle=0.08 | reward=0.562
INFO:simulation:steering=-0.65 | throttle=0.08 | reward=0.562
INFO:simulation:steering=-0.62 | throttle=0.09 | reward=0.556
INFO:simulation:steering=-0.58 | throttle=0.10 | reward=0.550
INFO:simulation:steering=-0.52 | throttle=0.11 | reward=0.632
INFO:simulation:steering=-0.47 | throttle=0.11 | reward=0.682
INFO:simulation:steering=-0.43 | throttle=0.12 | reward=0.696


INFO:simulation:steering=-0.39 | throttle=0.13 | reward=0.730
INFO:simulation:steering=-0.43 | throttle=0.12 | reward=0.674
INFO:simulation:steering=-0.47 | throttle=0.12 | reward=0.796


Starting Optuna Bayesian Optimization: 60 trials × 3 episodes each
Search space: Kp∈[0.3,2.5], Ki∈[0,0.3], Kd∈[0,0.5], α∈[0.3,1.0], bias∈[-0.2,0.2], brake∈[0,0.35]



INFO:simulation:steering=-0.51 | throttle=0.11 | reward=0.041
INFO:simulation:steering=-0.55 | throttle=0.10 | reward=0.054
INFO:simulation:steering=-0.58 | throttle=0.10 | reward=0.080
INFO:simulation:steering=-0.59 | throttle=0.09 | reward=0.093
INFO:simulation:steering=-0.59 | throttle=0.09 | reward=0.103
INFO:simulation:steering=-0.60 | throttle=0.09 | reward=0.125
INFO:simulation:steering=-0.59 | throttle=0.09 | reward=0.136
INFO:simulation:steering=-0.58 | throttle=0.09 | reward=0.146
INFO:simulation:steering=-0.57 | throttle=0.10 | reward=0.166
INFO:simulation:steering=-0.55 | throttle=0.10 | reward=0.176
INFO:simulation:steering=-0.53 | throttle=0.10 | reward=0.197
INFO:simulation:steering=-0.52 | throttle=0.11 | reward=0.219
INFO:simulation:steering=-0.50 | throttle=0.11 | reward=0.000
INFO:simulation:steering=-0.13 | throttle=0.18 | reward=0.121
INFO:simulation:steering=-0.16 | throttle=0.19 | reward=0.134
INFO:simulation:steering=-0.16 | throttle=0.19 | reward=0.153
INFO:sim

KeyboardInterrupt: 